# Pair diagnostics
Exploratory price-level regressions for PETR and BBDC. Full-sample coefficients are descriptive and must not be used as historical trading signals. These cells download market data.

In [ ]:


import yfinance as yf
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import statsmodels.api as sm

tickers = ["PETR3.SA", "PETR4.SA"]

data = yf.download(
    tickers,
    start="2020-01-01",
    end="2026-11-15",
    auto_adjust=True,
    progress=False
)

prices = data["Close"][tickers].dropna()

X = prices["PETR3.SA"]
Y = prices["PETR4.SA"]

print(prices.head())
print("Observations:", len(prices))

plt.figure(figsize=(12, 5))
plt.plot(X.index, X, label="PETR3")
plt.plot(Y.index, Y, label="PETR4")
plt.title("PETR3 vs PETR4 — Adjusted prices")
plt.xlabel("Date")
plt.ylabel("Adjusted price")
plt.legend()
plt.grid(alpha=0.3)
plt.show()


In [ ]:


X_reg = sm.add_constant(X)

model = sm.OLS(Y, X_reg).fit()

alpha = model.params["const"]
beta = model.params["PETR3.SA"]

print(f"Estimated alpha: {alpha:.4f}")
print(f"Hedge ratio (beta): {beta:.4f}")
print(f"R²: {model.rsquared:.4f}")

spread = Y - alpha - beta * X

plt.figure(figsize=(12, 5))
plt.plot(spread.index, spread, label="Residual spread")
plt.axhline(0, linestyle="--", label="Zero")
plt.title("Estimated spread: PETR4 - alpha - beta × PETR3")
plt.xlabel("Date")
plt.ylabel("Spread")
plt.legend()
plt.grid(alpha=0.3)
plt.show()


In [ ]:

from statsmodels.tsa.stattools import coint

stat, pvalue, critical_values = coint(Y, X)

print(f"Test statistic: {stat:.4f}")
print(f"p-value: {pvalue:.6f}")

print("\nCritical values:")
for level, value in zip(["1%", "5%", "10%"], critical_values):
    print(f"{level}: {value:.4f}")


In [ ]:

from statsmodels.tsa.stattools import adfuller
import pandas as pd

def print_adf(series, name):
    result = adfuller(
        series.dropna(),
        regression="c",
        autolag="AIC"
    )

    print(f"\n--- {name} ---")
    print(f"ADF statistic: {result[0]:.4f}")
    print(f"p-value: {result[1]:.6f}")
    print(f"Lags used: {result[2]}")

    if result[1] < 0.05:
        print("Reject the unit-root null at 5%.")
    else:
        print("Do not reject the unit-root null at 5%.")

print_adf(X, "PETR3 - Price")
print_adf(Y, "PETR4 - Price")

print_adf(X.diff(), "PETR3 - First difference")
print_adf(Y.diff(), "PETR4 - First difference")


In [ ]:

import matplotlib.pyplot as plt
import pandas as pd

spread = Y - alpha - beta * X

window = 60

rolling_mean = spread.rolling(window).mean()
rolling_std = spread.rolling(window).std()

zscore = (spread - rolling_mean) / rolling_std

fig, axes = plt.subplots(2, 1, figsize=(13, 8), sharex=True)

axes[0].plot(spread.index, spread, label="Spread")
axes[0].plot(rolling_mean.index, rolling_mean,
             label="60-session rolling mean", linestyle="--")
axes[0].set_title("PETR3 / PETR4 — Estimated spread")
axes[0].legend()
axes[0].grid(alpha=0.3)

axes[1].plot(zscore.index, zscore, label="Z-score")
axes[1].axhline(0, color="black", linestyle="--")
axes[1].axhline(2, color="red", linestyle="--")
axes[1].axhline(-2, color="green", linestyle="--")
axes[1].set_title("Rolling spread z-score")
axes[1].legend()
axes[1].grid(alpha=0.3)

plt.tight_layout()
plt.show()


In [ ]:
bank_prices = yf.download(["BBDC3.SA", "BBDC4.SA"], start="2020-01-01", end="2026-11-15", auto_adjust=True, progress=False)["Close"].dropna()

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from statsmodels.tsa.stattools import coint

pair = bank_prices[["BBDC3.SA", "BBDC4.SA"]].dropna()

X = pair["BBDC3.SA"]
Y = pair["BBDC4.SA"]

window = 252
step = 5  

dates = []
pvalues = []
statistics = []

for end in range(window, len(pair) + 1, step):
    x_window = X.iloc[end-window:end]
    y_window = Y.iloc[end-window:end]

    stat, pvalue, _ = coint(
        y_window,
        x_window,
        trend="c",
        autolag="aic"
    )

    dates.append(pair.index[end - 1])
    pvalues.append(pvalue)
    statistics.append(stat)

rolling_result = pd.DataFrame({
    "pvalue": pvalues,
    "statistic": statistics
}, index=dates)

plt.figure(figsize=(13, 5))

plt.plot(
    rolling_result.index,
    rolling_result["pvalue"],
    label="p-value Engle-Granger"
)

plt.axhline(
    0.05,
    color="red",
    linestyle="--",
    label="5% significance level"
)

plt.title("BBDC3 / BBDC4 - Rolling cointegration test")
plt.ylabel("p-value")
plt.xlabel("Date")
plt.ylim(0, 1)
plt.legend()
plt.grid(alpha=0.3)
plt.show()

print(
    "Share of windows with p < 0.05:",
    f"{(rolling_result['pvalue'] < 0.05).mean():.2%}"
)


In [ ]:

import pandas as pd
import matplotlib.pyplot as plt
import statsmodels.api as sm

pair = bank_prices[["BBDC3.SA", "BBDC4.SA"]].dropna()

X = pair["BBDC3.SA"]
Y = pair["BBDC4.SA"]

window = 252
betas = []
alphas = []
dates = []

for end in range(window, len(pair) + 1):
    x_window = X.iloc[end-window:end]
    y_window = Y.iloc[end-window:end]

    model = sm.OLS(
        y_window,
        sm.add_constant(x_window)
    ).fit()

    betas.append(model.params.iloc[1])
    alphas.append(model.params.iloc[0])
    dates.append(pair.index[end - 1])

rolling_beta = pd.Series(betas, index=dates)
rolling_alpha = pd.Series(alphas, index=dates)

plt.figure(figsize=(13, 5))
plt.plot(rolling_beta, label="Hedge ratio (beta)")
plt.axhline(
    rolling_beta.mean(),
    color="red",
    linestyle="--",
    label="Mean beta"
)
plt.title("BBDC3 / BBDC4 — Rolling hedge ratio")
plt.xlabel("Date")
plt.ylabel("Beta")
plt.legend()
plt.grid(alpha=0.3)
plt.show()

print("Mean beta:", round(rolling_beta.mean(), 4))
print("Minimum beta:", round(rolling_beta.min(), 4))
print("Maximum beta:", round(rolling_beta.max(), 4))
print("Beta standard deviation:", round(rolling_beta.std(), 4))
